# CarDekho Used Car Scraper — Kolkata

Same overall flow as the UrbanNest 99acres scraper: paginate through search results, pull the summary fields from each card, then visit each car's detail page for the extra specs.

**Before running this**: every selector below marked `# VERIFY` is a placeholder. CarDekho is a React app and its class names are frequently hashed/auto-generated (unlike 99acres' stable `tupleNew__*` classes), so they WILL differ from what's here by the time you run this. See the setup checklist in the chat response for how to find the real ones in ~10 minutes with DevTools.

In [9]:
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import os
from datetime import datetime
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


## Driver setup

Same headless config as UrbanNest, plus `webdriver-manager` so you don't have to manually download/match a chromedriver binary — it fetches the right version for whatever Chrome you have installed.

In [10]:
def get_chrome_options():
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124 Safari/537.36"
    )
    # Reduces the chance of basic bot-detection flagging headless Chrome
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)
    return chrome_options


def create_driver():
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=get_chrome_options())
    # Extra stealth: patch the navigator.webdriver flag
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"},
    )
    return driver

In [11]:
def scroll_page(driver, steps=6, pause=0.8):
    """Incrementally scroll to the bottom to trigger lazy-loaded content, then back to top."""
    last_height = driver.execute_script("return document.body.scrollHeight")
    for i in range(1, steps + 1):
        driver.execute_script(f"window.scrollTo(0, {last_height} * {i} / {steps});")
        time.sleep(pause)
    # a fresh height read + one more scroll to bottom, in case new content extended the page
    new_height = driver.execute_script("return document.body.scrollHeight")
    driver.execute_script(f"window.scrollTo(0, {new_height});")
    time.sleep(pause)
    driver.execute_script("window.scrollTo(0, 0);")  # back to top before reading elements
    time.sleep(0.5)

## Safe-extraction helpers

Identical pattern to UrbanNest — never let one missing element kill the whole row.

In [12]:
def safe_find(driver_or_elem, by, value, default=""):
    try:
        return driver_or_elem.find_element(by, value).text.strip()
    except Exception:
        return default


def safe_find_attr(driver_or_elem, by, value, attr, default=""):
    try:
        return driver_or_elem.find_element(by, value).get_attribute(attr)
    except Exception:
        return default


def safe_find_list(driver_or_elem, by, value):
    try:
        return [e.text for e in driver_or_elem.find_elements(by, value) if e.text.strip()]
    except Exception:
        return []

## Detail-page scraper

Visits an individual car's page for specs the listing card doesn't show (engine cc, mileage, seats, insurance, color, RTO, number of owners, etc.). Every selector here is a `# VERIFY` placeholder — fill in after inspecting an actual detail page.

In [13]:
def car_details_collect(link):
    driver = create_driver()
    try:
        driver.get(link)
        time.sleep(random.uniform(3, 5))

        details = {}

        # Fallback price/km from the detail page's top card - used when the listing
        # card (a different template, e.g. a "similar/certified" widget) has no price/specs
        details["Price (detail page)"] = safe_find(driver, By.CSS_SELECTOR, "span.vdpAtfCard__priceValue")
        details["KM Driven (detail page)"] = safe_find(driver, By.CSS_SELECTOR, "span.vdpAtfCard__km")

        details["EMI"] = safe_find(driver, By.CSS_SELECTOR, "span.vdpAtfCard__emiText")

        specs = safe_find_list(
            driver, By.CSS_SELECTOR, "div.vdpAtfCard__specs > span:not(.vdpAtfCard__dot)"
        )
        if len(specs) >= 3:
            details["Fuel Type (detail page)"] = specs[0]
            details["Transmission (detail page)"] = specs[1]
            details["Owner"] = specs[2]

        # Quick Insights loads asynchronously - explicitly wait for it instead of guessing a sleep
        try:
            scroll_page(driver, steps=4)
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.vdpQuickInsights_row"))
            )
            rows = driver.find_elements(By.CSS_SELECTOR, "div.vdpQuickInsights_row")
            for row in rows:
                key = safe_find(row, By.CSS_SELECTOR, "span.vdpQuickInsights__rowK")
                value = safe_find(row, By.CSS_SELECTOR, "span.vdpQuickInsights__rowV")
                if key:
                    details[key] = value
        except Exception:
            # Quick Insights genuinely absent for this listing, or didn't load in time - move on
            pass

    except Exception as e:
        print(f"\u26a0\ufe0f Detail page error at {link}: {e}")
        details = {}
    finally:
        driver.quit()

    print("Details collected for:", link)
    return details

## URL generation

CarDekho's Kolkata used-car listing base is `https://www.cardekho.com/used-cars+in+kolkata`. Confirm the pagination parameter yourself (commonly `?pageNo=` or `&page=` on this kind of site) by clicking to page 2 in a real browser and copying the resulting URL.

In [14]:
# Confirmed via browser: pagination is PATH-based, e.g. .../used-cars+in+kolkata/page-2
base_url = "https://www.cardekho.com/used-cars+in+kolkata"
num_pages = 30  # ~1300+ listed cars; adjust once you see how many render per page
urls = [base_url] + [f"{base_url}/page-{i}" for i in range(2, num_pages + 1)]

## Output setup

In [15]:
base_dir = "output"
backup_dir = os.path.join(base_dir, "cardekho_backups")
main_file = os.path.join(base_dir, "cardekho_kolkata_scraped_data.csv")
os.makedirs(backup_dir, exist_ok=True)

## Main scraper loop

Same structure as UrbanNest: for each listing page, grab every car card's summary fields, follow its link for extra detail-page specs, then save (main file + timestamped backup) after every page — so a crash halfway through doesn't cost you the whole run.

A few differences worth keeping deliberately, versus copying UrbanNest verbatim:
- **Randomized sleep** (`random.uniform`) instead of a fixed `time.sleep(5)` — fixed intervals are one of the easiest bot-detection signals to key off of.
- **One shared `driver` per listing page**, reused across detail-page visits within that page, instead of spinning up a brand-new Chrome instance per car (UrbanNest's `prop_details_collect` opens a new driver every single property, which is correctness-safe but slow at scale â€” up to you which you prefer for ~1,300 cars).

In [16]:
data = []

for page_num, url in enumerate(urls, start=1):
    driver = create_driver()
    try:
        driver.get(url)
        time.sleep(random.uniform(4, 6))
        scroll_page(driver)  # trigger lazy-loaded card content before reading

        # Confirmed via DevTools: card container is div.NewUcExCard
        cars = driver.find_elements(By.CSS_SELECTOR, "div.NewUcExCard")
        print(f"Found {len(cars)} cars on {url}")

        for car in cars:
            try:
                # Confirmed via DevTools inspection (screenshots)
                title_elem = car.find_element(By.CSS_SELECTOR, "h3.title a")
                full_title = title_elem.get_attribute("title")  # e.g. "2022 Maruti Suzuki Dzire ZXI Plus BSVI"
                link = title_elem.get_attribute("href")  # Selenium resolves this to an absolute URL already

                # First token of the title attribute is the year, if numeric
                first_token = full_title.split()[0] if full_title else ""
                year = first_token if first_token.isdigit() else ""
                model_name = full_title[len(first_token):].strip() if year else full_title

                price = safe_find(car, By.CSS_SELECTOR, "div.Price p").replace("\u20b9", "").strip()

                # dotsDetails has 3 direct-child divs: [km driven, transmission, fuel type]
                detail_divs = car.find_elements(By.XPATH, './/div[contains(@class,"dotsDetails")]/div')
                km_driven = detail_divs[0].text.strip() if len(detail_divs) > 0 else ""
                transmission = detail_divs[1].text.strip() if len(detail_divs) > 1 else ""
                fuel_type = detail_divs[2].text.strip() if len(detail_divs) > 2 else ""

                location = safe_find(car, By.CSS_SELECTOR, "div.distanceText span")

                # Bonus: CarDekho's own AI-generated highlight text ("2nd owner, 80k km \u2014 papers are clean...")
                ai_highlights = " | ".join(safe_find_list(car, By.CSS_SELECTOR, "div.reasonsMain"))

                more = car_details_collect(link) if link else {}

                # Fallback to detail-page price/km if the listing card template lacked them
                if not price:
                    price = more.get("Price (detail page)", "").replace("\u20b9", "").strip()
                if not km_driven:
                    km_driven = more.get("KM Driven (detail page)", "")

                entry = {
                    "Title": model_name,
                    "Link": link,
                    "Price": price,
                    "Year": year,
                    "KM Driven": km_driven,
                    "Fuel Type": fuel_type,
                    "Transmission": transmission,
                    "Location": location,
                    "AI Highlights": ai_highlights,
                    **more,
                }
                data.append(entry)
            except Exception as inner_e:
                print(f"\u26a0\ufe0f Car parse error: {inner_e}")

    except Exception as page_e:
        print(f"\u274c Page error at {url}: {page_e}")
    finally:
        driver.quit()

    print(f"Saving {len(data)} entries to file...")
    df = pd.DataFrame(data)
    df.to_csv(main_file, index=False)

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    backup_path = os.path.join(backup_dir, f"cardekho_data_{timestamp}.csv")
    df.to_csv(backup_path, index=False)

    print(f"\U0001f501 Scraping: {url} ({page_num}/{len(urls)})")
    time.sleep(random.uniform(2, 4))  # brief pause between pages

print("\U0001f389 All pages scraped and saved to 'cardekho_kolkata_scraped_data.csv'")

Found 68 cars on https://www.cardekho.com/used-cars+in+kolkata
Details collected for: https://www.cardekho.com/used-car-details/used-Mg-hector-sharp-dct-cars-Kolkata_ffc47d5d-a9c8-4d6d-a6a4-d30ba3e0f437.htm?adId=29723&adType=41
Details collected for: https://www.cardekho.com/used-car-details/used-Maruti-alto-k10-vxi-cars-Kolkata_d6c56954-9fcc-4c17-8093-f57f9307debc.htm
Details collected for: https://www.cardekho.com/used-car-details/used-Honda-wr-v-i-vtec-vx-cars-Kolkata_81053a31-d2a9-4f1a-bf74-18d7477ab465.htm
Details collected for: https://www.cardekho.com/used-car-details/used-Maruti-swift-dzire-vxi-cars-Kolkata_4a89c039-79c5-42f6-8af8-56dce0f687ad.htm
Details collected for: https://www.cardekho.com/used-car-details/used-Hyundai-verna-sx-cars-Kolkata_a8631b74-b794-454c-8b15-22648e6db5e6.htm?adId=31023&adType=41
Details collected for: https://www.cardekho.com/used-car-details/used-Hyundai-creta-sx-o-turbo-dct-cars-Kolkata_2d089dcf-6d13-466c-af99-64f5197bf386.htm
Details collected for

KeyboardInterrupt: 